In [1]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Read the tickers from a file

In [3]:
tickers = pd.read_csv('Tickers.csv', header=None)[0]

tickers = tickers.dropna().astype(str).str.strip().str.upper()
tickers = tickers[tickers != ''].drop_duplicates()

raw_info = {}
for ticker in tickers:
    try:
        raw_info[ticker] = yf.Ticker(ticker).info
    except Exception:
        raw_info[ticker] = {}

In [4]:
type_s= []
currency = []
market = []
sector = []

for ticker in tickers: 
    try: 
        tick = raw_info.get(ticker)
        type_s.append(tick.get('quoteType'))
        currency.append(tick.get('currency'))
        market.append(tick.get('market'))
        sector.append(tick.get('sector'))
    except:
        type_s.append(None)
        currency.append(None)
        market.append(None)
        sector.append(None)

info = {
    'Type' : type_s,
    'Currency' : currency,
    'Market' : market,
    'Sector' : sector
}

all_stocks = pd.DataFrame(info, index=tickers)
all_stocks.index.name = 'Ticker'
all_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AGN,NONE,None,None,None
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials


check if it is eligible

In [5]:
mask = all_stocks['Type'] == 'EQUITY'
mask1 = all_stocks['Currency'].isin(['USD', 'CAD'])
mask2 = all_stocks['Market'].isin(['us_market', 'ca_market'])
mask3 = all_stocks['Sector'].notna() & (all_stocks['Sector'] != '')

eligible_stocks = all_stocks[mask & mask1 & mask2 & mask3].copy()
eligible_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials
BAC,EQUITY,USD,us_market,Financial Services


check for volume

In [6]:
global_start='2025-10-01'
global_end='2026-11-28'
tickers_list = eligible_stocks.index.tolist().copy()
hist_data = yf.download(tickers_list, start=global_start, end=global_end, group_by='ticker')

[*********************100%***********************]  35 of 35 completed


CHECK VOLUME

In [7]:
start = '2025-10-01'
end = '2026-09-30'
min_avg_volume = 5000

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[start:end].copy()
    history = history.dropna(subset=['Close', 'Volume'])

    if history.empty:
        eligible_stocks.loc[ticker, 'Avg_Volume'] = None
        continue

    history['Month'] = history.index.strftime('%Y-%m')
    
    month_counts = history.groupby('Month').size()
    valid_months = month_counts[month_counts >= 18].index
    history = history[history['Month'].isin(valid_months)]

    eligible_stocks.loc[ticker, 'Avg_Volume'] = history.Volume.mean()

mask = eligible_stocks['Avg_Volume'] >= min_avg_volume
eligible_stocks = eligible_stocks[mask].copy()

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume
Ticker,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07
ACN,EQUITY,USD,us_market,Technology,6.116120e+06
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06
BA,EQUITY,USD,us_market,Industrials,7.140954e+06
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07


take the closing price

In [8]:
buy_date = '2026-11-27'
end_date = '2026-11-28'

eligible_stocks['Close'] = np.nan

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[buy_date:end_date]
    if buy_date in history.index:
        eligible_stocks.loc[ticker, 'Close'] = history.loc[buy_date, 'Close']

mask = eligible_stocks['Close'].notna() & (eligible_stocks['Close'] > 0)
eligible_stocks = eligible_stocks[mask].copy()

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close
Ticker,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000


convert to CAD

In [9]:
usdcad = yf.Ticker('USDCAD=X').history(start=buy_date, end=end_date)
rate = float(usdcad['Close'].dropna().iloc[-1])

for ticker in eligible_stocks.index.tolist():
    price = eligible_stocks.loc[ticker, 'Close']
    if eligible_stocks.loc[ticker, 'Currency'] == 'USD':
        eligible_stocks.loc[ticker, 'Close_CAD'] = price * rate
    else :
        eligible_stocks.loc[ticker, 'Close_CAD'] = price

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD
Ticker,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858


PORTFOLIO A

calculate Beta using S&P 500 as a benchmark

In [10]:
market_hist = yf.download('^GSPC', start=global_start, end=global_end)
market_close = market_hist['Close']['^GSPC']

for ticker in eligible_stocks.index.tolist():
    both = pd.DataFrame({'stock': hist_data[ticker]['Close'], 'market': market_close}).dropna()
    both_returns = both.pct_change().dropna()

    manual_beta = both_returns['stock'].cov(both_returns['market']) / both_returns['market'].var()
    eligible_stocks.loc[ticker, 'Beta'] = manual_beta

eligible_stocks

[*********************100%***********************]  1 of 1 completed


,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta
Ticker,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.659460
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,-0.112865
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,0.014736
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,0.167356
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,0.024237
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.486927
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,0.952157
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,1.198185
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,0.738764


If we couldn't calculate a beta ourselves, use Yahoo's beta instead. If Yahoo doesn't have one either, just assume average risk (1.0)

In [11]:
for ticker in eligible_stocks.index.tolist():
    beta = eligible_stocks.loc[ticker, 'Beta']

    if pd.isna(beta):
        yahoo_beta = raw_info[ticker].get('beta')
        if yahoo_beta is None:
            beta = 1.0
        else:
            beta = yahoo_beta
        eligible_stocks.loc[ticker, 'Beta'] = beta

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta
Ticker,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.659460
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,-0.112865
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,0.014736
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,0.167356
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,0.024237
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.486927
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,0.952157
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,1.198185
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,0.738764


1 year returns

In [12]:
one_year_ago = (pd.Timestamp(buy_date) - pd.DateOffset(years=1)).strftime('%Y-%m-%d')

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker]
    old_price = history['Close'].dropna().asof(pd.Timestamp(one_year_ago))
    new_price = eligible_stocks.loc[ticker, 'Close']

    if pd.notna(old_price) and old_price > 0 and pd.notna(new_price):
        eligible_stocks.loc[ticker, 'Return %'] = (new_price - old_price) / old_price * 100
    else:
        eligible_stocks.loc[ticker, 'Return %'] = None

# Drop any stock we couldn't compute a 1-year return for 
eligible_stocks = eligible_stocks.dropna(subset=['Return %']).copy()
eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta,Return %
Ticker,,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.659460,29.601649
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,-0.112865,29.351981
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,0.014736,-16.918916
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,0.167356,-3.524663
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,0.024237,2.812190
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.486927,11.512537
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,0.952157,-12.507702
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,1.198185,-24.582963
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,0.738764,-2.375913


Combine Beta and Return into one Score

In [13]:
beta_norm = (eligible_stocks['Beta'] - eligible_stocks['Beta'].min()) / (eligible_stocks['Beta'].max() - eligible_stocks['Beta'].min())
return_norm = (eligible_stocks['Return %'] - eligible_stocks['Return %'].min()) / (eligible_stocks['Return %'].max() - eligible_stocks['Return %'].min())

eligible_stocks['Score'] = (beta_norm + return_norm) / 2
eligible_stocks.sort_values('Score', ascending=False)

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta,Return %,Score
Ticker,,,,,,,,,,
BB.TO,EQUITY,CAD,ca_market,Technology,3.406802e+06,13.390000,13.390000,1.723391,99.850757,0.929705
TXN,EQUITY,USD,us_market,Technology,7.674424e+06,307.409576,430.760749,1.458318,87.066811,0.832623
CAT,EQUITY,USD,us_market,Industrials,2.807785e+06,713.309998,999.532780,1.750823,36.485816,0.703669
QCOM,EQUITY,USD,us_market,Technology,1.412401e+07,184.169998,258.070055,2.092906,0.173336,0.636220
TD.TO,EQUITY,CAD,ca_market,Financial Services,5.614886e+06,176.570007,176.570007,0.650010,59.194990,0.577135
C,EQUITY,USD,us_market,Financial Services,1.269748e+07,122.019730,170.981369,1.320057,24.057326,0.576363
SHOP.TO,EQUITY,CAD,ca_market,Technology,1.943639e+06,232.929993,232.929993,1.860194,-4.992456,0.573097
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.486927,11.512537,0.562325
BIIB,EQUITY,USD,us_market,Healthcare,1.408238e+06,252.729996,354.140438,0.317312,68.005058,0.545997


Split into a growth bucket (Technology + Communication Services, target 40%) and everything else (target 60%). If there aren't enough growth stocks, use fewer and lower the target so we never break the 15% per-stock cap

In [14]:
#select sectors the portfolio is going to focus on
growth_sectors = ['Technology', 'Communication Services']

mask = eligible_stocks['Sector'].isin(growth_sectors)
growth_stocks = eligible_stocks[mask].sort_values('Score', ascending=False)
other_stocks = eligible_stocks[~mask].sort_values('Score', ascending=False)

# total stock count
n_total = 15
# use 5 selected sector stocks, or fewer if the ticker file doesn't have 5
n_growth = min(5, len(growth_stocks))   
n_other = n_total - n_growth

#take the n_growth number of the best stocks in the selected sectors
selected_growth = growth_stocks.head(n_growth).copy()
selected_other = other_stocks.head(n_other).copy()

# Quick sanity check: make sure no OTHER single sector is secretly dominating this bucket
selected_other['Sector'].value_counts()

Sector
Financial Services    4
Healthcare            3
Industrials           2
Consumer Cyclical     1
Name: count, dtype: int64

Weight each bucket by Score, then cap any single stock at 15% and hand the extra weight to the rest of its bucket

In [15]:
tolerance = 0.0000001

def cap_weights(df, cap=0.15):
    while (df['Weight'] > cap + tolerance).any():
        excess = 0
        for ticker in df.index:
            if df.loc[ticker, 'Weight'] > cap:
                excess = excess + (df.loc[ticker, 'Weight'] - cap)
                df.loc[ticker, 'Weight'] = cap
    
        not_capped = df[df['Weight'] < cap]
        for ticker in not_capped.index:
            df.loc[ticker, 'Weight'] = df.loc[ticker, 'Weight'] + excess / len(not_capped)

    return df

#for the 40% sector cap
def cap_sectors(df, cap=0.40):
    tries = 0
    while (df.groupby('Sector')['Weight'].sum() > cap + tolerance).any() and tries < 50:
        tries = tries + 1
        sector_weights = df.groupby('Sector')['Weight'].sum()
        excess = 0

        for sec in sector_weights.index:
            if sector_weights[sec] > cap:
                excess = excess + (sector_weights[sec] - cap)
                mask = df['Sector'] == sec
                df.loc[mask, 'Weight'] = df.loc[mask, 'Weight'] * (cap / sector_weights[sec])

        under_sectors = sector_weights[sector_weights < cap - tolerance].index
        not_capped = df[df['Sector'].isin(under_sectors) & (df['Weight'] < 0.15 - tolerance)]
        if len(not_capped) == 0:
            break

        for ticker in not_capped.index:
            df.loc[ticker, 'Weight'] = df.loc[ticker, 'Weight'] + excess / len(not_capped)

        df = cap_weights(df)

    return df

# can't ask for more than the 15% cap allows with this many stocks
target_growth = min(0.40, n_growth * 0.15)
target_other = 1 - target_growth

selected_growth['Weight'] = selected_growth['Score'] / selected_growth['Score'].sum() * target_growth
selected_other['Weight'] = selected_other['Score'] / selected_other['Score'].sum() * target_other

selected_growth = cap_weights(selected_growth)
selected_other = cap_weights(selected_other)

portfolio_A = pd.concat([selected_growth, selected_other])

portfolio_A = cap_sectors(portfolio_A)

print('Total weight:', portfolio_A['Weight'].sum())
print('Growth sector weight:', selected_growth['Weight'].sum())
portfolio_A[['Sector', 'Beta', 'Return %', 'Score', 'Weight']].sort_values('Weight', ascending=False)

Total weight: 1.0
Growth sector weight: 0.39999999999999997


,Sector,Beta,Return %,Score,Weight
Ticker,,,,,
BB.TO,Technology,1.723391,99.850757,0.929705,0.108024
TXN,Technology,1.458318,87.066811,0.832623,0.096744
CAT,Industrials,1.750823,36.485816,0.703669,0.077861
QCOM,Technology,2.092906,0.173336,0.636220,0.073924
SHOP.TO,Technology,1.860194,-4.992456,0.573097,0.066589
TD.TO,Financial Services,0.650010,59.194990,0.577135,0.063860
C,Financial Services,1.320057,24.057326,0.576363,0.063774
AMZN,Consumer Cyclical,1.486927,11.512537,0.562325,0.062221
BIIB,Healthcare,0.317312,68.005058,0.545997,0.060414


In [16]:
fee_rate = 0.001 
def calc_fee_cad(shares_num):
    return min(shares_num * fee_rate, 2.15) * rate

In [17]:
budget = 1000000
def buy_stocks(portfolio, eligible):
    # first pass: split the budget by weight, ignoring fees
    for ticker in portfolio.index.tolist():
        money_for_stock = portfolio.loc[ticker, 'Weight'] * budget
        stock_n = money_for_stock / eligible.loc[ticker, 'Close_CAD']
        portfolio.loc[ticker, 'Shares'] = stock_n

    # add up what the fees would come out to with these share counts
    total_fees = 0
    for ticker in portfolio.index.tolist():
        fee = calc_fee_cad(portfolio.loc[ticker, 'Shares'])
        total_fees = total_fees + fee

    # second pass: reduce the budget by the fees, then size shares again
    investable = budget - total_fees
    for ticker in portfolio.index.tolist():
        money_for_stock = portfolio.loc[ticker, 'Weight'] * investable
        stock_n = money_for_stock / eligible.loc[ticker, 'Close_CAD']
        portfolio.loc[ticker, 'Shares'] = stock_n

    # final value + fee per stock, and a check that we spent close to budget
    total_value = 0
    total_fees_final = 0
    for ticker in portfolio.index.tolist():
        value = portfolio.loc[ticker, 'Shares'] * eligible.loc[ticker, 'Close_CAD']
        fee = calc_fee_cad(portfolio.loc[ticker, 'Shares'])
        portfolio.loc[ticker, 'Value'] = value
        portfolio.loc[ticker, 'Fee'] = fee
        total_value = total_value + value
        total_fees_final = total_fees_final + fee

    print('Total spent:', total_value + total_fees_final)
    print('Budget:', budget)
    print('Leftover:', budget - total_value - total_fees_final)

    return portfolio

portfolio_A = buy_stocks(portfolio_A, eligible_stocks)
portfolio_A

Total spent: 999999.9999600013
Budget: 1000000
Leftover: 3.9998674253638455e-05


,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta,Return %,Score,Weight,Shares,Value,Fee
Ticker,,,,,,,,,,,,,,
BB.TO,EQUITY,CAD,ca_market,Technology,3.406802e+06,13.390000,13.390000,1.723391,99.850757,0.929705,0.108024,8067.480776,108023.570360,3.012709
TXN,EQUITY,USD,us_market,Technology,7.674424e+06,307.409576,430.760749,1.458318,87.066811,0.832623,0.096744,224.587533,96743.493746,0.314706
QCOM,EQUITY,USD,us_market,Technology,1.412401e+07,184.169998,258.070055,2.092906,0.173336,0.636220,0.073924,286.445885,73923.105386,0.401385
SHOP.TO,EQUITY,CAD,ca_market,Technology,1.943639e+06,232.929993,232.929993,1.860194,-4.992456,0.573097,0.066589,285.874648,66588.779716,0.400585
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.659460,29.601649,0.470929,0.054718,112.586287,54717.847702,0.157763
CAT,EQUITY,USD,us_market,Industrials,2.807785e+06,713.309998,999.532780,1.750823,36.485816,0.703669,0.077861,77.896500,77860.104919,0.109153
TD.TO,EQUITY,CAD,ca_market,Financial Services,5.614886e+06,176.570007,176.570007,0.650010,59.194990,0.577135,0.063860,361.665548,63859.288430,0.506787
C,EQUITY,USD,us_market,Financial Services,1.269748e+07,122.019730,170.981369,1.320057,24.057326,0.576363,0.063774,372.987519,63773.916522,0.522652
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.486927,11.512537,0.562325,0.062221,175.437729,62220.555461,0.245834


In [18]:
Portfolio_Final_A_data = {
    'Ticker' : portfolio_A.index,
    'Price' : portfolio_A.Close.values,
    'Currency' : portfolio_A.Currency.values,
    'Shares' : portfolio_A.Shares.values,
    'Value' : portfolio_A.Value.values,
    # to reduce fees from the total value of the invested shares
    'Weight' : (portfolio_A.Value / portfolio_A.Value.sum()).values
}

Portfolio_Final_A = pd.DataFrame(Portfolio_Final_A_data, index = range(1, len(portfolio_A) + 1))
Portfolio_Final_A

,Ticker,Price,Currency,Shares,Value,Weight
1,BB.TO,13.390000,CAD,8067.480776,108023.570360,0.108024
2,TXN,307.409576,USD,224.587533,96743.493746,0.096744
3,QCOM,184.169998,USD,286.445885,73923.105386,0.073924
4,SHOP.TO,232.929993,CAD,285.874648,66588.779716,0.066589
5,AAPL,346.836487,USD,112.586287,54717.847702,0.054718
6,CAT,713.309998,USD,77.896500,77860.104919,0.077861
7,TD.TO,176.570007,CAD,361.665548,63859.288430,0.063860
8,C,122.019730,USD,372.987519,63773.916522,0.063774
9,AMZN,253.100006,USD,175.437729,62220.555461,0.062221
10,BIIB,252.729996,USD,170.593199,60413.950436,0.060414


TASK 2 

We use the same trailing 1-year window as the rest of the notebook (one_year_ago to buy_date), so this stays consistent with how Beta and Return % were already calculated.

In [19]:
# Daily USD/CAD rate over the whole window (not just one day) so we can convert
# each day's USD prices into CAD, instead of only converting the final price
fx_hist = yf.Ticker('USDCAD=X').history(start=one_year_ago, end=end_date)
fx_rate_daily = fx_hist['Close']

#to fix the timezore issues
def to_naive_dates(series):
    series = series.copy()
    series.index = pd.to_datetime(series.index.date)
    return series

fx_rate_daily = to_naive_dates(fx_rate_daily)

For each stock: build a daily CAD-converted price series, then turn that into a daily return series. Do this once for the strategy weights and once for equal weights, using exactly the same return series both times.

In [20]:
returns_df = pd.DataFrame()
def compare_portfolios(portfolio, eligible):
    # build CAD daily prices and daily returns for the stocks in this portfolio
    returns_data = {}
    for ticker in portfolio.index.tolist():
        prices = hist_data[ticker]['Close'].loc[one_year_ago:buy_date]
        prices = to_naive_dates(prices)

        if eligible.loc[ticker, 'Currency'] == 'USD':
            prices_cad = prices * fx_rate_daily
        else:
            prices_cad = prices

        returns_data[ticker] = prices_cad.ffill().pct_change(fill_method=None)

    returns_df = pd.DataFrame(returns_data).dropna()

    strategy_returns = pd.Series(0, index=returns_df.index)
    for ticker in portfolio.index.tolist():
        strategy_returns = strategy_returns + returns_df[ticker] * portfolio.loc[ticker, 'Weight']

    # equal-weighted daily return: same stocks, weight = 1/n for every one
    equal_weight = 1 / len(portfolio)
    equal_returns = pd.Series(0, index=returns_df.index)
    for ticker in portfolio.index.tolist():
        equal_returns = equal_returns + returns_df[ticker] * equal_weight

    trading_days = 252
    comparison = pd.DataFrame({
        'Avg Daily Return %': [strategy_returns.mean() * 100, equal_returns.mean() * 100],
        'Daily Volatility %': [strategy_returns.std() * 100, equal_returns.std() * 100],
        'Annualized Return %': [strategy_returns.mean() * trading_days * 100, equal_returns.mean() * trading_days * 100],
        'Annualized Volatility %': [strategy_returns.std() * (trading_days ** 0.5) * 100, equal_returns.std() * (trading_days ** 0.5) * 100]
    }, index=['Portfolio A - Strategy Weights', 'Portfolio A - Equal Weights'])

    return comparison

comparison_A = compare_portfolios(portfolio_A, eligible_stocks)
comparison_A

,Avg Daily Return %,Daily Volatility %,Annualized Return %,Annualized Volatility %
Portfolio A - Strategy Weights,0.168902,1.178592,42.563241,18.709570
Portfolio A - Equal Weights,0.157145,1.070760,39.600610,16.997796


Explanations

TASK 3

Checks Portfolio_Final_A against the six rules the assignment lists, and prints a pass/fail for each one plus an overall result.

In [21]:
def validate_portfolio(portfolio_final, eligible, budget):
    portfolio_final = portfolio_final.copy()
   
    print("TASK 3 - VALIDATION CHECKS")
    print("-" * 40)

    # 1. every ticker in the final portfolio is eligible and unique
    all_valid = True
    for ticker in portfolio_final['Ticker']:
        if ticker not in eligible.index:
            all_valid = False
        else:
            row = eligible.loc[ticker]
            if (row['Type'] != 'EQUITY'
                or row['Currency'] not in ['USD', 'CAD']
                or row['Market'] not in ['us_market', 'ca_market']
                or pd.isna(row['Avg_Volume']) or row['Avg_Volume'] < 5000):
                all_valid = False
    if not portfolio_final['Ticker'].is_unique:
        all_valid = False
    print("All tickers eligible & unique:", all_valid)

    # 2. stock count between 10 and 20
    n_stocks = len(portfolio_final)
    count_ok = 10 <= n_stocks <= 20
    print("Stock count (10-20):", count_ok, "-", n_stocks, "stocks")

    # 3. no stock over 15%
    max_weight = portfolio_final['Weight'].max()
    max_weight_ok = max_weight <= 0.15 + tolerance
    print("Max individual weight <= 15%:", max_weight_ok, "-", round(max_weight * 100, 2), "%")

    # 4. no sector over 40%
    portfolio_final['Sector'] = portfolio_final['Ticker'].map(eligible['Sector'])
    sector_weights = portfolio_final.groupby('Sector')['Weight'].sum()
    max_sector_weight = sector_weights.max()
    max_sector_ok = max_sector_weight <= 0.40 + tolerance
    print("Max sector weight <= 40%:", max_sector_ok, "-", round(max_sector_weight * 100, 2), "% (" + sector_weights.idxmax() + ")")

    # 5. weights sum to 100%
    weight_sum = portfolio_final['Weight'].sum()
    weight_sum_ok = abs(weight_sum - 1) < tolerance
    print("Weights sum to 100%:", weight_sum_ok, "-", round(weight_sum * 100, 2), "%")

    # 6. total spent + fees within budget
    total_value = portfolio_final['Value'].sum()
    total_fees = 0
    for shares in portfolio_final['Shares']:
        total_fees = total_fees + calc_fee_cad(shares)
    total_spent = total_value + total_fees
    budget_ok = total_spent <= budget + tolerance
    print("Spend + fees within budget:", budget_ok, "-", "$" + str(round(total_spent, 2)), "of", "$" + str(budget))

    print("-" * 40)
    all_passed = all_valid and count_ok and max_weight_ok and max_sector_ok and weight_sum_ok and budget_ok
    print("ALL CHECKS PASSED:", all_passed)

    return all_passed

validate_portfolio(Portfolio_Final_A, eligible_stocks, budget)

TASK 3 - VALIDATION CHECKS
----------------------------------------
All tickers eligible & unique: True
Stock count (10-20): True - 15 stocks
Max individual weight <= 15%: True - 10.8 %
Max sector weight <= 40%: True - 40.0 % (Technology)
Weights sum to 100%: True - 100.0 %
Spend + fees within budget: True - $1000000.0 of $1000000
----------------------------------------
ALL CHECKS PASSED: True


np.True_

In [22]:
group_number = '3'
Stocks_A = Portfolio_Final_A[['Ticker', 'Shares']]
Stocks_A.to_csv(f'Stocks_Group_{group_number}_A.csv', index=False)